In [ ]:
from __future__ import annotations

from dataclasses import dataclass
from pathlib import Path
from typing import Any

import pandas as pd

try:
    import duckdb
except Exception as exc:
    raise RuntimeError("Install duckdb in this notebook environment to run this database notebook.") from exc

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

@dataclass
class DuckDBConfig:
    chunks_path: Path = ROOT / "data" / "processed_reports" / "chunks" / "text_chunks.parquet"
    manifest_path: Path = ROOT / "data" / "processed_reports" / "page_or_sheet_manifest.parquet"
    structured_path: Path = ROOT / "data" / "processed_reports" / "tables" / "structured_long.parquet"
    metric_dictionary_path: Path = ROOT / "data" / "processed_reports" / "tables" / "metric_dictionary.parquet"
    semantic_metrics_path: Path = ROOT / "data" / "processed_reports" / "tables" / "semantic_metrics.parquet"
    vector_records_path: Path = ROOT / "data" / "processed_reports" / "semantic_vector_records.parquet"
    duckdb_path: Path = ROOT / "storage" / "insightviewer_semantic.duckdb"

CFG = DuckDBConfig()
CFG.duckdb_path.parent.mkdir(parents=True, exist_ok=True)
print(f"DuckDB target: {CFG.duckdb_path}")

## **Load Semantic Artifacts**

In [ ]:
def read_dataframe(path: Path) -> pd.DataFrame:
    if path.exists():
        if path.suffix == ".parquet":
            return pd.read_parquet(path)
        if path.suffix == ".csv":
            return pd.read_csv(path)

    csv_fallback = path.with_suffix(".csv")
    if csv_fallback.exists():
        return pd.read_csv(csv_fallback)

    print(f"Missing artifact: {path}")
    return pd.DataFrame()


tables = {
    "metric_dictionary": read_dataframe(CFG.metric_dictionary_path),
    "semantic_metrics": read_dataframe(CFG.semantic_metrics_path),
    "semantic_vector_records": read_dataframe(CFG.vector_records_path),
    "page_or_sheet_manifest": read_dataframe(CFG.manifest_path),
    "text_chunks": read_dataframe(CFG.chunks_path),
    "structured_long": read_dataframe(CFG.structured_path),
}

for table_name, frame in tables.items():
    print(table_name, frame.shape)

## **Materialize Tables and Views**

In [ ]:
def write_table(conn: duckdb.DuckDBPyConnection, table_name: str, frame: pd.DataFrame) -> None:
    conn.register("_frame", frame)
    conn.execute(f"CREATE OR REPLACE TABLE {table_name} AS SELECT * FROM _frame")
    conn.unregister("_frame")


conn = duckdb.connect(str(CFG.duckdb_path))
for table_name, frame in tables.items():
    write_table(conn, table_name, frame)

conn.execute("""
    CREATE OR REPLACE VIEW metric_timeseries AS
    SELECT
        entity,
        ticker,
        metric_canonical,
        metric_display_name,
        fiscal_year_semantic AS fiscal_year,
        fiscal_quarter_semantic AS fiscal_quarter,
        period_label,
        period_sort,
        units,
        currency,
        SUM(value) AS value,
        COUNT(*) AS source_row_count
    FROM semantic_metrics
    WHERE value IS NOT NULL
    GROUP BY
        entity, ticker, metric_canonical, metric_display_name, fiscal_year_semantic,
        fiscal_quarter_semantic, period_label, period_sort, units, currency
""")

conn.execute("""
    CREATE OR REPLACE VIEW retrieval_catalog AS
    SELECT
        vector_id,
        content_kind,
        document,
        metadata_json
    FROM semantic_vector_records
""")

print(f"Wrote DuckDB semantic database: {CFG.duckdb_path}")

## **Query Examples**

In [ ]:
def query_metric_timeseries(metric: str, ticker: str | None = None) -> pd.DataFrame:
    sql = """
        SELECT *
        FROM metric_timeseries
        WHERE metric_canonical = ?
    """
    params: list[Any] = [metric]
    if ticker:
        sql += " AND ticker = ?"
        params.append(ticker)
    sql += " ORDER BY period_sort"
    return conn.execute(sql, params).df()


query_metric_timeseries("revenue", ticker="MSFT").head(20)

In [ ]:
conn.close()